In [1]:
import os
import random
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.nn import CTCLoss
from tqdm import tqdm
import editdistance

In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [3]:
BATCH_SIZE = 16
IMG_HEIGHT = 32
IMG_WIDTH = 128
EPOCHS_PHASE1 = 5   # only classifier
EPOCHS_PHASE2 = 15  # full fine-tuning
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ARABIC_CHARS = 'ابتثجحخدذرزسشصضطظعغفقكلمنهوي0123456789'


In [4]:
class LabelConverter:
    def __init__(self, chars):
        self.chars = chars
        self.char2idx = {c: i+1 for i, c in enumerate(chars)}  # 0 reserved for CTC blank
        self.idx2char = {i+1: c for i, c in enumerate(chars)}
        self.blank = 0

    def encode(self, text):
        return torch.tensor([self.char2idx[c] for c in text], dtype=torch.long)

    def decode(self, preds):
        preds = preds.argmax(2).permute(1, 0).cpu().numpy()
        decoded = []
        for seq in preds:
            text, prev = '', -1
            for p in seq:
                if p != prev and p != self.blank:
                    text += self.idx2char.get(p, '')
                prev = p
            decoded.append(text)
        return decoded

converter = LabelConverter(ARABIC_CHARS)

In [5]:
class ArabicDataset(Dataset):
    def __init__(self, txt_path, img_dir):
        self.samples = []
        with open(txt_path, 'r', encoding='utf-8') as f:
            for line in f:
                name, label = line.strip().split(maxsplit=1)
                self.samples.append((os.path.join(img_dir, name), label))
        self.transform = transforms.Compose([
            transforms.Grayscale(),
            transforms.Resize((IMG_HEIGHT, IMG_WIDTH)),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")
        image = self.transform(image)
        return {'image': image, 'label': converter.encode(label), 'text': label}

In [6]:
def collate_fn(batch):
    images = torch.stack([b['image'] for b in batch])
    labels = [b['label'] for b in batch]
    label_lengths = torch.tensor([len(l) for l in labels])
    texts = [b['text'] for b in batch]
    return images, torch.cat(labels), label_lengths, texts


In [7]:
class CRNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, 1, 1), nn.ReLU(),
            nn.Conv2d(256, 256, 3, 1, 1), nn.ReLU(), nn.MaxPool2d((2,1)),
            nn.Conv2d(256, 512, 3, 1, 1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.Conv2d(512, 512, 3, 1, 1), nn.BatchNorm2d(512), nn.ReLU(), nn.MaxPool2d((2,1)),
            nn.Conv2d(512, 512, 2, 1, 0), nn.ReLU()
        )
        self.rnn = nn.Sequential(
            nn.LSTM(512, 256, bidirectional=True, batch_first=True),
            nn.LSTM(512, 256, bidirectional=True, batch_first=True)
        )
        self.fc = nn.Linear(512, num_classes + 1)

    def forward(self, x):
        x = self.cnn(x)         # [B, 512, 1, W]
        x = x.squeeze(2).permute(0, 2, 1)  # [B, W, 512]
        x, _ = self.rnn(x)
        x = self.fc(x)          # [B, W, C]
        return x.permute(1, 0, 2)  # [W, B, C] for CTC


In [8]:
train_loader = DataLoader(ArabicDataset(r"C:\Users\Raihan\OneDrive\Desktop\DPIIT HACKATHON\train_list.txt", r"C:\Users\Raihan\OneDrive\Desktop\DPIIT HACKATHON\images"),
                          batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(ArabicDataset(r"C:\Users\Raihan\OneDrive\Desktop\DPIIT HACKATHON\val_list.txt", r"C:\Users\Raihan\OneDrive\Desktop\DPIIT HACKATHON\images"),
                        batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)


In [10]:
model = CRNN(num_classes=len(ARABIC_CHARS)).to(DEVICE)
checkpoint = torch.load(r"C:\Users\Raihan\OneDrive\Desktop\DPIIT HACKATHON\crnn_ctc_checkpoint.pth", map_location=DEVICE)
filtered = {k: v for k, v in checkpoint['model_state_dict'].items() if not k.startswith("fc.")}
model.load_state_dict(filtered, strict=False)
print("✅ Pretrained weights loaded (except FC)")


✅ Pretrained weights loaded (except FC)


In [11]:
for name, param in model.named_parameters():
    if not name.startswith("fc"):
        param.requires_grad = False

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
criterion = CTCLoss(blank=0, zero_infinity=True)

print("🔧 Training FC Layer Only...")
for epoch in range(1, EPOCHS_PHASE1 + 1):
    model.train()
    total_loss = 0
    for images, labels, label_lengths, _ in tqdm(train_loader, desc=f"[Phase 1] Epoch {epoch}"):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        preds = model(images)
        input_lengths = torch.full((images.size(0),), preds.size(0), dtype=torch.long).to(DEVICE)

        loss = criterion(preds, labels, input_lengths, label_lengths.to(DEVICE))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch} Loss: {total_loss:.4f}")


🔧 Training FC Layer Only...


[Phase 1] Epoch 1:   0%|          | 0/8846 [00:00<?, ?it/s]


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Raihan\\OneDrive\\Desktop\\DPIIT'